# Northline Cycles — Agents Workshop setup

This notebook prepares **everything you need for the workshop** in your own Databricks Free Edition workspace:

| What | Name | Used by |
|---|---|---|
| Schema | `agents_workshop` | everything |
| 4 tables | `customers`, `products`, `orders`, `support_tickets` | both agents |
| Volume with 4 policy documents | `policy_docs` | Customer Care agent |
| The same policies as a table, one row per section | `policy_documents` | read by the `get_policy` tool |
| 3 tools (Unity Catalog SQL functions) | `check_return_eligibility`, `customer_summary`, `get_policy` | Customer Care agent |

**How to run it:** click **Run all** at the top of the page. It takes about 3–5 minutes on serverless compute.
You can leave the widgets at their default values.

> Northline Cycles is a fictional online bike and gear retailer based in Montréal that ships across Canada. All data is synthetic.

In [ ]:
# 1. Configuration: the defaults work on Databricks Free Edition
dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0], "Catalog")
dbutils.widgets.text("schema", "agents_workshop", "Schema")
dbutils.widgets.dropdown("reset", "yes", ["yes", "no"], "Start from scratch")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
RESET = dbutils.widgets.get("reset") == "yes"

if CATALOG.lower() == "hive_metastore":
    raise ValueError(
        "This workshop needs a Unity Catalog catalog. "
        "Set the 'catalog' widget to 'workspace' (the default catalog in Free Edition) and run again."
    )

FQ = f"{CATALOG}.{SCHEMA}"
VOLUME_NAME = "policy_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"


def sql_str(text):
    """Quote a Python string as a SQL string literal."""
    return "'" + text.replace("\\", "\\\\").replace("'", "\\'") + "'"


print(f"Catalog: {CATALOG}\nSchema:  {FQ}\nVolume:  {VOLUME_PATH}\nReset:   {RESET}")

In [ ]:
# 2. Create the schema and the volume that will hold the policy documents
if RESET:
    spark.sql(f"DROP SCHEMA IF EXISTS {FQ} CASCADE")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ} COMMENT "
          + sql_str("Northline Cycles: synthetic data for the Databricks agents workshop."))
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQ}.{VOLUME_NAME} COMMENT "
          + sql_str("Northline Cycles customer policies: returns, warranty, shipping and rewards."))
print(f"Schema {FQ} and volume {VOLUME_NAME} are ready.")

In [ ]:
# 3. Generate the synthetic data (deterministic: everyone gets the same dataset)
import random
from datetime import date, timedelta
from decimal import Decimal, ROUND_HALF_UP

rng = random.Random(2026)
TODAY = date.today()


def money(x):
    return Decimal(str(x)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


# ---------- Products ----------
BIKE_CATEGORIES = {"Road Bikes", "Mountain Bikes", "E-Bikes", "City Bikes"}
PRODUCT_LIST = [
    # (name, category, brand, price, on_sale)
    ("Northline Aero 105", "Road Bikes", "Northline", 2499, False),
    ("Aurora Sprint SL", "Road Bikes", "Aurora", 3299, False),
    ("Granite Endurance 2", "Road Bikes", "Granite", 1899, True),
    ("Northline Gravel GRX", "Road Bikes", "Northline", 2799, False),
    ("Granite Trail 29", "Mountain Bikes", "Granite", 2199, False),
    ("Summit Ridge XC", "Mountain Bikes", "Summit Gear", 3499, False),
    ("Tundra Fat 4.8", "Mountain Bikes", "Tundra", 1999, True),
    ("Granite Enduro Pro", "Mountain Bikes", "Granite", 4599, False),
    ("Voltway City E", "E-Bikes", "Voltway", 2899, False),
    ("Voltway Commuter 500", "E-Bikes", "Voltway", 3499, False),
    ("Voltway Cargo Max", "E-Bikes", "Voltway", 5299, False),
    ("Voltway Trail E+", "E-Bikes", "Voltway", 5999, False),
    ("Northline Metro 7", "City Bikes", "Northline", 899, False),
    ("Aurora Step-Through", "City Bikes", "Aurora", 749, True),
    ("Northline Fold 20", "City Bikes", "Northline", 1099, False),
    ("Summit Gear Road Helmet MIPS", "Helmets", "Summit Gear", 179, False),
    ("Summit Gear Trail Helmet", "Helmets", "Summit Gear", 149, False),
    ("Northline Commuter Helmet", "Helmets", "Northline", 89, False),
    ("Summit Gear Kids Helmet", "Helmets", "Summit Gear", 59, False),
    ("Northline U-Lock Pro", "Accessories", "Northline", 89, False),
    ("Aurora LED Light Set", "Accessories", "Aurora", 69, False),
    ("Northline Floor Pump", "Accessories", "Northline", 59, False),
    ("Tundra Panniers 40L", "Accessories", "Tundra", 149, False),
    ("Northline Bottle Cage", "Accessories", "Northline", 15, False),
    ("Aurora Bike Computer GPS", "Accessories", "Aurora", 249, False),
    ("Tundra Winter Gloves", "Apparel", "Tundra", 49, True),
    ("Northline Rain Jacket", "Apparel", "Northline", 159, False),
    ("Summit Gear Padded Shorts", "Apparel", "Summit Gear", 89, False),
    ("Northline Merino Jersey", "Apparel", "Northline", 119, True),
    ("Granite Tubeless Tire 29x2.4", "Components", "Granite", 79, False),
    ("Northline Chain 11-Speed", "Components", "Northline", 45, False),
    ("Aurora Disc Brake Pads", "Components", "Aurora", 29, False),
    ("Voltway Spare Battery 500Wh", "Components", "Voltway", 799, False),
    ("Northline Carbon Wheelset", "Components", "Northline", 1299, False),
]
products = [
    {"product_id": f"P{i:03d}", "product_name": n, "category": c, "brand": b,
     "unit_price": money(p), "on_sale": s}
    for i, (n, c, b, p, s) in enumerate(PRODUCT_LIST, start=1)
]
PRODUCT_BY_ID = {p["product_id"]: p for p in products}
PRODUCT_BY_NAME = {p["product_name"]: p for p in products}
CATEGORY_WEIGHT = {"Road Bikes": 0.05, "Mountain Bikes": 0.05, "E-Bikes": 0.04, "City Bikes": 0.04,
                   "Helmets": 0.15, "Accessories": 0.30, "Apparel": 0.20, "Components": 0.17}
CATEGORY_SIZE = {c: sum(1 for p in products if p["category"] == c) for c in CATEGORY_WEIGHT}
PRODUCT_WEIGHTS = [CATEGORY_WEIGHT[p["category"]] / CATEGORY_SIZE[p["category"]] for p in products]

# ---------- Customers ----------
FIRST = ["Olivia", "Liam", "Emma", "Noah", "Charlotte", "William", "Amelia", "Lucas", "Sophie", "Benjamin",
         "Chloe", "Ethan", "Léa", "Mathis", "Aisha", "Omar", "Priya", "Arjun", "Mei", "Wei", "Fatou",
         "Kofi", "Sofia", "Mateo", "Hannah", "Jacob", "Zoé", "Gabriel", "Nadia", "Samuel", "Isabelle",
         "Thomas", "Rosalie", "Felix", "Yara", "Daniel", "Grace", "Hugo", "Inès", "Raphaël"]
LAST = ["Tremblay", "Gagnon", "Roy", "Côté", "Bouchard", "Gauthier", "Morin", "Lavoie", "Fortin", "Gagné",
        "Smith", "Brown", "Wilson", "MacDonald", "Campbell", "Anderson", "Taylor", "Martin", "Nguyen",
        "Chen", "Singh", "Patel", "Diallo", "Mensah", "Haddad", "Rossi", "Silva", "Kim", "Park", "Wong"]
CITIES = [("Montréal", "QC", 0.24), ("Québec City", "QC", 0.08), ("Laval", "QC", 0.05),
          ("Sherbrooke", "QC", 0.03), ("Gatineau", "QC", 0.03), ("Toronto", "ON", 0.17),
          ("Ottawa", "ON", 0.07), ("Hamilton", "ON", 0.03), ("Vancouver", "BC", 0.10),
          ("Victoria", "BC", 0.03), ("Calgary", "AB", 0.06), ("Edmonton", "AB", 0.04),
          ("Winnipeg", "MB", 0.03), ("Halifax", "NS", 0.02), ("Moncton", "NB", 0.01),
          ("St. John's", "NL", 0.005), ("Whitehorse", "YT", 0.005)]

N_CUSTOMERS = 1200
customers = []
for i in range(N_CUSTOMERS):
    city, prov, _ = rng.choices(CITIES, weights=[w for *_, w in CITIES])[0]
    # 85% joined before the 12-month order window, the rest during it
    if rng.random() < 0.85:
        signup = TODAY - timedelta(days=rng.randint(366, 3 * 365))
    else:
        signup = TODAY - timedelta(days=rng.randint(30, 365))
    customers.append({"customer_id": f"C{1001 + i}", "first_name": rng.choice(FIRST),
                      "last_name": rng.choice(LAST), "city": city, "province": prov,
                      "signup_date": signup})

# Two named customers used in the demo scenarios
DEMO_CUSTOMERS = {
    "C1042": {"first_name": "Maya", "last_name": "Chen", "city": "Montréal", "province": "QC",
              "signup_date": TODAY - timedelta(days=800)},
    "C1107": {"first_name": "Liam", "last_name": "Gagnon", "city": "Québec City", "province": "QC",
              "signup_date": TODAY - timedelta(days=500)},
}
for c in customers:
    c.update(DEMO_CUSTOMERS.get(c["customer_id"], {}))
CUSTOMER_BY_ID = {c["customer_id"]: c for c in customers}
# A few customers buy a lot. The two demo customers only get their hand-written orders.
CUSTOMER_WEIGHTS = [0 if c["customer_id"] in DEMO_CUSTOMERS else rng.paretovariate(1.6) for c in customers]

# ---------- Orders ----------
SEASON = {1: 0.5, 2: 0.5, 3: 0.8, 4: 1.2, 5: 1.6, 6: 1.8, 7: 1.8, 8: 1.6, 9: 1.2, 10: 0.9, 11: 0.8, 12: 1.0}
WINDOW = [TODAY - timedelta(days=d) for d in range(0, 365)]
DAY_WEIGHTS = [SEASON[d.month] for d in WINDOW]
FAST_PROVINCES = {"QC", "ON"}


def delivery_lead(province):
    if province in FAST_PROVINCES:
        return rng.randint(2, 4)
    if province in {"YT", "NT", "NU"}:
        return rng.randint(7, 12)
    return rng.randint(4, 7)


N_ORDERS = 3000
orders = []
for i in range(N_ORDERS):
    order_date = rng.choices(WINDOW, weights=DAY_WEIGHTS)[0]
    for _ in range(20):  # the customer must have signed up before ordering
        cust = rng.choices(customers, weights=CUSTOMER_WEIGHTS)[0]
        if cust["signup_date"] <= order_date:
            break
    prod = rng.choices(products, weights=PRODUCT_WEIGHTS)[0]
    qty = 1 if prod["category"] in BIKE_CATEGORIES else rng.choices([1, 2, 3], weights=[70, 22, 8])[0]
    promo = rng.choices([None, "SPRING10", "WELCOME10", "LOYAL10"], weights=[85, 6, 5, 4])[0]
    total = money(float(prod["unit_price"]) * qty * (0.9 if promo else 1.0))
    lead = delivery_lead(cust["province"])
    days_ago = (TODAY - order_date).days
    if days_ago <= 1:
        status, delivered = "Processing", None
    elif order_date + timedelta(days=lead) > TODAY:
        status, delivered = "In transit", None
    else:
        r = rng.random()
        status = "Cancelled" if r < 0.03 else ("Returned" if r < 0.08 else "Delivered")
        delivered = None if status == "Cancelled" else order_date + timedelta(days=lead)
    orders.append({"order_id": f"ORD-{20001 + i}", "customer_id": cust["customer_id"],
                   "product_id": prod["product_id"], "quantity": qty, "unit_price": prod["unit_price"],
                   "order_total": total, "order_date": order_date, "delivery_date": delivered,
                   "status": status,
                   "channel": rng.choices(["Website", "Mobile app", "Partner store"], weights=[60, 30, 10])[0],
                   "promo_code": promo})


def demo_order(order_id, customer_id, product_name, status, delivered_days_ago=None, ordered_days_ago=None):
    prod = PRODUCT_BY_NAME[product_name]
    delivered = TODAY - timedelta(days=delivered_days_ago) if delivered_days_ago is not None else None
    ordered = TODAY - timedelta(days=ordered_days_ago if ordered_days_ago is not None else delivered_days_ago + 3)
    return {"order_id": order_id, "customer_id": customer_id, "product_id": prod["product_id"],
            "quantity": 1, "unit_price": prod["unit_price"], "order_total": prod["unit_price"],
            "order_date": ordered, "delivery_date": delivered, "status": status,
            "channel": "Website", "promo_code": None}


# Hand-written orders that the workshop questions rely on
orders += [
    demo_order("ORD-30001", "C1042", "Voltway Commuter 500", "Delivered", delivered_days_ago=10),
    demo_order("ORD-30002", "C1042", "Aurora Sprint SL", "Delivered", delivered_days_ago=40),
    demo_order("ORD-30003", "C1107", "Summit Gear Road Helmet MIPS", "Delivered", delivered_days_ago=5),
    demo_order("ORD-30004", "C1107", "Tundra Winter Gloves", "Delivered", delivered_days_ago=12),
    demo_order("ORD-30005", "C1042", "Northline U-Lock Pro", "In transit", ordered_days_ago=2),
]
ORDER_BY_ID = {o["order_id"]: o for o in orders}

# ---------- Support tickets ----------
SUMMARIES = {
    "Return request": ["Customer wants to return the {p}: wrong size.",
                       "Customer changed their mind about the {p} and asks how to return it.",
                       "The {p} does not match the description; customer requests a return.",
                       "Customer received the {p} as a gift and wants a refund."],
    "Warranty claim": ["The {p} developed a defect after a few weeks of normal use.",
                       "Customer reports a crack on the {p}; asks if the warranty covers it.",
                       "The {p} stopped working; customer requests a repair or replacement."],
    "Delivery issue": ["The {p} has not arrived yet; customer asks for tracking details.",
                       "The box containing the {p} arrived damaged.",
                       "Courier marked the {p} as delivered but the customer did not receive it."],
    "Billing": ["Customer was charged twice for the {p}.",
                "Promo code was not applied to the {p} order.",
                "Customer asks for an invoice for the {p}."],
    "Product question": ["Customer asks which size of the {p} fits a 175 cm rider.",
                         "Customer asks if the {p} is compatible with their current bike.",
                         "Customer asks about maintenance for the {p}."],
}


def ticket_status(opened):
    age = (TODAY - opened).days
    if age <= 2:
        return "Open"
    if age <= 10:
        return rng.choice(["Open", "In progress", "In progress", "Resolved"])
    return rng.choices(["Resolved", "In progress", "Open"], weights=[92, 5, 3])[0]


tickets = []


def add_ticket(customer_id, order, category, opened, status=None, priority=None, summary=None):
    status = status or ticket_status(opened)
    prod_name = PRODUCT_BY_ID[order["product_id"]]["product_name"] if order else "product"
    if priority is None:
        priority = "High" if category in ("Delivery issue", "Warranty claim") and rng.random() < 0.4 \
            else rng.choices(["Low", "Normal", "High"], weights=[25, 60, 15])[0]
    score = rng.choices([1, 2, 3, 4, 5], weights=[5, 8, 15, 35, 37])[0] if status == "Resolved" else None
    tickets.append({"ticket_id": f"T-{5001 + len(tickets)}", "customer_id": customer_id,
                    "order_id": order["order_id"] if order else None, "opened_date": opened,
                    "category": category, "status": status, "priority": priority,
                    "summary": summary or rng.choice(SUMMARIES[category]).format(p=prod_name),
                    "satisfaction_score": score})


for o in orders:
    if o["order_id"].startswith("ORD-3"):
        continue
    cat = PRODUCT_BY_ID[o["product_id"]]["category"]
    if o["status"] == "Returned" and rng.random() < 0.85:
        add_ticket(o["customer_id"], o, "Return request",
                   min(TODAY, o["delivery_date"] + timedelta(days=rng.randint(1, 20))))
    elif o["status"] == "In transit" and rng.random() < 0.15:
        add_ticket(o["customer_id"], o, "Delivery issue", TODAY - timedelta(days=rng.randint(0, 2)))
    elif o["status"] == "Delivered":
        r = rng.random()
        opened = min(TODAY, o["delivery_date"] + timedelta(days=rng.randint(1, 60)))
        if r < 0.035 and (cat in BIKE_CATEGORIES or cat == "Components"):
            add_ticket(o["customer_id"], o, "Warranty claim", opened)
        elif r < 0.07:
            add_ticket(o["customer_id"], o, "Delivery issue",
                       min(TODAY, o["delivery_date"] + timedelta(days=rng.randint(0, 3))))
        elif r < 0.10:
            add_ticket(o["customer_id"], o, "Billing", opened)
        elif r < 0.13:
            add_ticket(o["customer_id"], o, "Product question", opened)

for _ in range(40):  # questions not linked to an order
    add_ticket(rng.choice(customers[200:])["customer_id"], None, "Product question",
               TODAY - timedelta(days=rng.randint(0, 300)),
               summary=rng.choice(["Customer asks for advice on choosing between a road bike and a gravel bike.",
                                   "Customer asks whether e-bikes can be shipped to their province.",
                                   "Customer asks when winter gloves will be back in stock.",
                                   "Customer asks how the Northline Rewards points work."]))

# Demo tickets
add_ticket("C1042", ORDER_BY_ID["ORD-30001"], "Return request", TODAY - timedelta(days=1), status="Open",
           priority="Normal",
           summary="Customer says the Voltway Commuter 500 is too large for her and wants to return it.")
add_ticket("C1107", ORDER_BY_ID["ORD-30003"], "Product question", TODAY, status="Open", priority="Low",
           summary="Customer tried the helmet on once and asks if he can still return it.")

print(f"{len(customers)} customers, {len(products)} products, {len(orders)} orders, {len(tickets)} tickets")

In [ ]:
# 4. Write the tables to Unity Catalog, with descriptions that help Genie understand the data
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, DateType,
                               DecimalType, BooleanType)


def col(name, dtype, comment, nullable=True):
    return StructField(name, dtype, nullable, metadata={"comment": comment})


TABLES = {
    "customers": (
        "One row per Northline Cycles customer.",
        StructType([
            col("customer_id", StringType(), "Unique customer identifier, for example C1042.", False),
            col("first_name", StringType(), "Customer first name."),
            col("last_name", StringType(), "Customer last name."),
            col("city", StringType(), "City of the shipping address."),
            col("province", StringType(), "Canadian province or territory code, for example QC, ON, BC."),
            col("signup_date", DateType(), "Date the customer created their account."),
        ]),
        customers,
    ),
    "products": (
        "Product catalogue. Prices are in Canadian dollars (CAD).",
        StructType([
            col("product_id", StringType(), "Unique product identifier, for example P010.", False),
            col("product_name", StringType(), "Product name as shown on the website."),
            col("category", StringType(), "One of: Road Bikes, Mountain Bikes, E-Bikes, City Bikes, Helmets, "
                                          "Accessories, Apparel, Components."),
            col("brand", StringType(), "Brand name."),
            col("unit_price", DecimalType(10, 2), "Current list price in CAD."),
            col("on_sale", BooleanType(), "True if the product is currently on sale. Sale items are "
                                          "refunded as store credit."),
        ]),
        products,
    ),
    "orders": (
        "One row per order. Each order contains a single product (with a quantity). Amounts in CAD.",
        StructType([
            col("order_id", StringType(), "Unique order identifier, for example ORD-30001.", False),
            col("customer_id", StringType(), "Customer who placed the order. Joins to customers.customer_id."),
            col("product_id", StringType(), "Product ordered. Joins to products.product_id."),
            col("quantity", IntegerType(), "Number of units ordered."),
            col("unit_price", DecimalType(10, 2), "Unit list price at the time of the order, in CAD."),
            col("order_total", DecimalType(10, 2), "Amount paid in CAD after any promo code (10% off). "
                                                   "Use this column for revenue."),
            col("order_date", DateType(), "Date the order was placed."),
            col("delivery_date", DateType(), "Date the order was delivered. Null if not delivered yet or cancelled."),
            col("status", StringType(), "One of: Processing, In transit, Delivered, Returned, Cancelled. "
                                        "Exclude Cancelled and Returned orders when computing revenue."),
            col("channel", StringType(), "Sales channel: Website, Mobile app or Partner store."),
            col("promo_code", StringType(), "Promo code used, if any (SPRING10, WELCOME10, LOYAL10)."),
        ]),
        orders,
    ),
    "support_tickets": (
        "Customer support tickets handled by the Northline Cycles customer care team.",
        StructType([
            col("ticket_id", StringType(), "Unique ticket identifier, for example T-5001.", False),
            col("customer_id", StringType(), "Customer who opened the ticket. Joins to customers.customer_id."),
            col("order_id", StringType(), "Related order, if any. Joins to orders.order_id."),
            col("opened_date", DateType(), "Date the ticket was opened."),
            col("category", StringType(), "One of: Return request, Warranty claim, Delivery issue, Billing, "
                                          "Product question."),
            col("status", StringType(), "One of: Open, In progress, Resolved."),
            col("priority", StringType(), "One of: Low, Normal, High."),
            col("summary", StringType(), "Short description of the customer request."),
            col("satisfaction_score", IntegerType(), "Customer satisfaction from 1 (bad) to 5 (great). "
                                                     "Only set for resolved tickets."),
        ]),
        tickets,
    ),
}

for name, (comment, schema, rows) in TABLES.items():
    df = spark.createDataFrame([tuple(r[f.name] for f in schema.fields) for r in rows], schema)
    (df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.{name}"))
    spark.sql(f"COMMENT ON TABLE {FQ}.{name} IS {sql_str(comment)}")
    print(f"{FQ}.{name}: {df.count()} rows")

# Primary and foreign keys let Genie discover the joins on its own.
# They are optional: if your workspace refuses them, the workshop still works.
CONSTRAINTS = [
    "ALTER TABLE {fq}.{t} ALTER COLUMN {c} SET NOT NULL",
    "ALTER TABLE {fq}.{t} ADD CONSTRAINT {t}_pk PRIMARY KEY ({c})",
]
FOREIGN_KEYS = [
    ("orders", "customer_id", "customers"), ("orders", "product_id", "products"),
    ("support_tickets", "customer_id", "customers"), ("support_tickets", "order_id", "orders"),
]
key_statements = []
for t, c in [("customers", "customer_id"), ("products", "product_id"),
             ("orders", "order_id"), ("support_tickets", "ticket_id")]:
    key_statements += [stmt.format(fq=FQ, t=t, c=c) for stmt in CONSTRAINTS]
for t, c, ref in FOREIGN_KEYS:
    key_statements.append(f"ALTER TABLE {FQ}.{t} ADD CONSTRAINT {t}_{c}_fk FOREIGN KEY ({c}) "
                          f"REFERENCES {FQ}.{ref}({c})")
failed = []
for stmt in key_statements:
    try:
        spark.sql(stmt)
    except Exception as e:  # noqa: BLE001
        failed.append(str(e).splitlines()[0][:100])
if not failed:
    print("Primary and foreign keys added.")
else:
    print(f"Note: {len(failed)} of {len(key_statements)} key statements were skipped ({failed[0]}). "
          "This is fine: you can define the joins in Genie instead.")

In [ ]:
# 5. Write the four policy documents into the volume (+ a fallback table with the same text)
POLICIES = {}

POLICIES["return_policy.md"] = """# Northline Cycles: Return and Refund Policy

## Return window
- Most items can be returned within **30 days** of the delivery date.
- **E-Bikes** can be returned within **15 days** of the delivery date, because of battery safety checks.
- The return window is the same for every Northline Rewards tier.
- Orders that have not been delivered yet cannot be returned. Orders with the status Processing can be cancelled by customer care; orders In transit must be delivered first.

## Condition of returned items
- Items must be unused, clean and in their original packaging with all accessories.
- **Helmets** can only be returned if they are unworn and still in the sealed hygiene packaging. A helmet that has been worn cannot be returned for safety reasons. This makes helmet returns conditional.
- Components that have already been installed on a bike (tires, chains, brake pads, wheelsets) cannot be returned, unless they are defective.
- Gift cards cannot be returned.

## Refund method
- Refunds go back to the **original payment method** within 5 to 10 business days after the return is received.
- Items that were **on sale** are refunded as **store credit** only.

## Return shipping fee
- A return shipping fee is deducted from the refund: **$15** for standard items and **$75** for bikes (Road Bikes, Mountain Bikes, E-Bikes and City Bikes), which need oversized shipping.
- The return shipping fee is **waived** when the item is defective or arrived damaged.
- The return shipping fee is also **waived for Gold members** of Northline Rewards.

## After the return window
- When the return window has passed, a product may still be covered by the warranty. See the Warranty Policy.

## How to start a return
1. Contact customer care with the order ID.
2. Customer care confirms eligibility and sends a prepaid return label.
3. The refund is issued once the warehouse has inspected the item (usually 3 business days).
"""

POLICIES["warranty_policy.md"] = """# Northline Cycles: Warranty Policy

## Coverage periods
- **Bike frames** (Northline, Aurora, Granite, Summit Gear and Tundra bikes): **5 years** against manufacturing defects.
- **Other bike parts** (fork, wheels, drivetrain, brakes) sold on a complete bike: **2 years**.
- **E-Bike motors and batteries** (Voltway): **2 years or 500 full charge cycles**, whichever comes first.
- **Spare batteries and components** sold separately: **1 year**.
- **Helmets**: **1 year** against manufacturing defects.
- **Apparel and accessories**: **1 year** against manufacturing defects.

## What the warranty does not cover
- Normal wear items: tires, tubes, chains, cassettes, brake pads, grips and saddles.
- Damage caused by crashes, misuse, racing or commercial use.
- Bikes assembled or modified by a shop that is not a Northline certified partner.
- Water damage caused by pressure washing an e-bike.

## Helmet crash replacement
- If a Northline or Summit Gear helmet is damaged in a crash within 2 years of purchase, the customer can buy a replacement at **40% off**.

## How to make a warranty claim
1. Open a support ticket in the Warranty claim category with the order ID.
2. Attach photos of the defect and, for e-bikes, the battery charge cycle count from the display.
3. Customer care answers within 2 business days. Approved claims are repaired or replaced at no cost, including shipping.
"""

POLICIES["shipping_policy.md"] = """# Northline Cycles: Shipping Policy

## Shipping costs
- Standard shipping is **free for orders over $100**. Otherwise it costs **$12.99**.
- Express shipping costs **$24.99** and is available in Quebec and Ontario only.
- Northline Rewards Silver and Gold members get free standard shipping on every order.

## Delivery times (after the order ships)
- Quebec (QC) and Ontario (ON): **2 to 4 business days**.
- Other provinces: **4 to 7 business days**.
- Territories (Yukon, Northwest Territories, Nunavut): **7 to 12 business days**.
- Orders are prepared and shipped within 1 business day.

## Bikes and e-bikes
- Bikes ship 95% assembled. Customers need to install the handlebar, pedals and front wheel.
- E-bike batteries ship separately when required by carrier rules and may arrive one day later.
- E-bikes cannot be shipped to the territories because of battery transport rules.

## Delivery problems
- If an order shows as delivered but was not received, customers must contact customer care within 7 days.
- Damaged boxes should be photographed before opening. Items that arrive damaged are replaced at no cost, and any return shipping fee is waived.
"""

POLICIES["rewards_program.md"] = """# Northline Rewards Program

## How tiers work
- The tier is based on the amount spent in the **last 12 months**, excluding cancelled and returned orders.
- **Bronze**: less than $1,000.
- **Silver**: from $1,000 to $2,999.99.
- **Gold**: $3,000 or more.

## Benefits by tier
- **Bronze**: earn 1 point per dollar spent; birthday coupon of $10.
- **Silver**: everything in Bronze, plus free standard shipping on every order and 5% off workshop services.
- **Gold**: everything in Silver, plus **free return shipping** (the return shipping fee is waived), free express shipping in Quebec and Ontario, and early access to sales.

## Points
- Customers earn 1 point per dollar spent.
- 500 points can be exchanged for $25 of store credit.
- Points expire after 18 months without any purchase.
"""

for filename, text in POLICIES.items():
    dbutils.fs.put(f"{VOLUME_PATH}/{filename}", text, overwrite=True)
    print(f"Wrote {VOLUME_PATH}/{filename}")

# The same policies as a table (one row per section). The get_policy tool reads it, so the agent
# works even where the "analyze files in volumes" preview is not available.
policy_rows = []
for filename, text in POLICIES.items():
    title = text.splitlines()[0].lstrip("# ").strip()
    for chunk in text.split("\n## ")[1:]:
        heading, _, body = chunk.partition("\n")
        policy_rows.append((len(policy_rows) + 1, title, heading.strip(), body.strip()))
policy_schema = StructType([
    col("section_number", IntegerType(), "Reading order of the section across all policies."),
    col("document", StringType(), "Policy document title."),
    col("section", StringType(), "Section heading inside the document."),
    col("content", StringType(), "Full text of the section."),
])
(spark.createDataFrame(policy_rows, policy_schema)
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.policy_documents"))
spark.sql(f"COMMENT ON TABLE {FQ}.policy_documents IS "
          + sql_str("Northline Cycles customer policies (returns, warranty, shipping, rewards), one row per section. "
                    "Search the content column to answer policy questions."))
print(f"{FQ}.policy_documents: {len(policy_rows)} sections")

In [ ]:
# 6. Create the three tools the Customer Care agent will call (Unity Catalog SQL functions)
BIKES = "('Road Bikes', 'Mountain Bikes', 'E-Bikes', 'City Bikes')"

spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.check_return_eligibility(
  target_order_id STRING COMMENT 'Order identifier, for example ORD-30001'
)
RETURNS TABLE (
  order_id STRING, customer_id STRING, product_name STRING, category STRING, status STRING,
  delivery_date DATE, days_since_delivery INT, return_window_days INT,
  eligible STRING, refund_method STRING, return_shipping_fee DECIMAL(10,2),
  estimated_refund DECIMAL(10,2), explanation STRING, fee_note STRING
)
COMMENT 'Checks whether an order can be returned under the Northline Cycles return policy and estimates the refund. Use it for any question about returning an order or about a refund amount. eligible is Yes, No or Conditional.'
RETURN
SELECT
  b.order_id, b.customer_id, b.product_name, b.category, b.status, b.delivery_date,
  b.days_since_delivery, b.return_window_days,
  CASE
    WHEN b.status <> 'Delivered' THEN 'No'
    WHEN b.days_since_delivery > b.return_window_days THEN 'No'
    WHEN b.category = 'Helmets' THEN 'Conditional'
    ELSE 'Yes'
  END AS eligible,
  CASE
    WHEN b.status <> 'Delivered' OR b.days_since_delivery > b.return_window_days THEN 'None'
    WHEN b.on_sale THEN 'Store credit'
    ELSE 'Original payment method'
  END AS refund_method,
  CASE
    WHEN b.status <> 'Delivered' OR b.days_since_delivery > b.return_window_days THEN CAST(0 AS DECIMAL(10,2))
    ELSE b.return_fee
  END AS return_shipping_fee,
  CASE
    WHEN b.status <> 'Delivered' OR b.days_since_delivery > b.return_window_days THEN CAST(0 AS DECIMAL(10,2))
    ELSE CAST(b.order_total - b.return_fee AS DECIMAL(10,2))
  END AS estimated_refund,
  CASE
    WHEN b.status = 'Returned' THEN 'This order has already been returned.'
    WHEN b.status = 'Cancelled' THEN 'This order was cancelled, so there is nothing to return.'
    WHEN b.status <> 'Delivered' THEN concat('The order status is ', b.status, '. A return can only start after delivery.')
    WHEN b.days_since_delivery > b.return_window_days THEN concat('The ', b.return_window_days, '-day return window has passed (delivered ', b.days_since_delivery, ' days ago). The warranty may still apply.')
    WHEN b.category = 'Helmets' THEN 'Helmets can only be returned if unworn and in the sealed hygiene packaging.'
    WHEN b.on_sale THEN 'Eligible. Sale items are refunded as store credit.'
    ELSE 'Eligible for a refund to the original payment method.'
  END AS explanation,
  'The return shipping fee is waived for defective items and for Gold members of Northline Rewards.' AS fee_note
FROM (
  SELECT
    o.order_id, o.customer_id, p.product_name, p.category, o.status, o.delivery_date,
    o.order_total, p.on_sale,
    CAST(datediff(current_date(), o.delivery_date) AS INT) AS days_since_delivery,
    CASE WHEN p.category = 'E-Bikes' THEN 15 ELSE 30 END AS return_window_days,
    CAST(CASE WHEN p.category IN {BIKES} THEN 75 ELSE 15 END AS DECIMAL(10,2)) AS return_fee
  FROM {FQ}.orders o
  JOIN {FQ}.products p ON o.product_id = p.product_id
  WHERE o.order_id = upper(trim(target_order_id))
) b
""")
print(f"Created {FQ}.check_return_eligibility")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.customer_summary(
  target_customer_id STRING COMMENT 'Customer identifier, for example C1042'
)
RETURNS TABLE (
  customer_id STRING, full_name STRING, city STRING, province STRING, customer_since DATE,
  total_orders BIGINT, spend_last_12_months DECIMAL(12,2), rewards_tier STRING,
  open_tickets BIGINT, last_order_date DATE
)
COMMENT 'Returns a one-row profile of a customer: order history, spend over the last 12 months, Northline Rewards tier (Bronze, Silver or Gold) and number of open support tickets. Use it whenever a question is about a specific customer.'
RETURN
SELECT
  c.customer_id,
  concat(c.first_name, ' ', c.last_name) AS full_name,
  c.city, c.province, c.signup_date AS customer_since,
  coalesce(o.total_orders, 0) AS total_orders,
  CAST(coalesce(o.spend_12m, 0) AS DECIMAL(12,2)) AS spend_last_12_months,
  CASE
    WHEN coalesce(o.spend_12m, 0) >= 3000 THEN 'Gold'
    WHEN coalesce(o.spend_12m, 0) >= 1000 THEN 'Silver'
    ELSE 'Bronze'
  END AS rewards_tier,
  coalesce(t.open_tickets, 0) AS open_tickets,
  o.last_order_date
FROM {FQ}.customers c
LEFT JOIN (
  SELECT customer_id,
         count(*) AS total_orders,
         sum(CASE WHEN order_date >= add_months(current_date(), -12)
                   AND status NOT IN ('Cancelled', 'Returned') THEN order_total ELSE 0 END) AS spend_12m,
         max(order_date) AS last_order_date
  FROM {FQ}.orders
  GROUP BY customer_id
) o ON o.customer_id = c.customer_id
LEFT JOIN (
  SELECT customer_id, count(*) AS open_tickets
  FROM {FQ}.support_tickets
  WHERE status <> 'Resolved'
  GROUP BY customer_id
) t ON t.customer_id = c.customer_id
WHERE c.customer_id = upper(trim(target_customer_id))
""")
print(f"Created {FQ}.customer_summary")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.get_policy(
  topic STRING COMMENT 'Policy topic: returns, warranty, shipping or rewards'
)
RETURNS TABLE (document STRING, section STRING, content STRING)
COMMENT 'Returns the full text of a Northline Cycles customer policy, one row per section, in reading order. Use it for any question about what the return and refund, warranty, shipping or rewards policy says. topic is one of: returns, warranty, shipping, rewards. Any other value returns every policy.'
RETURN
SELECT d.document, d.section, d.content
FROM {FQ}.policy_documents d
CROSS JOIN (
  SELECT CASE lower(trim(topic))
           WHEN 'returns' THEN 'return' WHEN 'return' THEN 'return'
           WHEN 'refund' THEN 'return' WHEN 'refunds' THEN 'return'
           WHEN 'warranty' THEN 'warranty' WHEN 'warranties' THEN 'warranty'
           WHEN 'shipping' THEN 'shipping' WHEN 'delivery' THEN 'shipping'
           WHEN 'rewards' THEN 'rewards' WHEN 'reward' THEN 'rewards' WHEN 'loyalty' THEN 'rewards'
         END AS k
) t
WHERE t.k IS NULL OR lower(d.document) LIKE concat('%', t.k, '%')
ORDER BY d.section_number
""")
print(f"Created {FQ}.get_policy")

In [ ]:
# 7. Smoke test: 5 demo orders, Maya Chen as a Gold member, and the 4 sections of the warranty policy
for t in ["customers", "products", "orders", "support_tickets", "policy_documents"]:
    print(f"{t:<18} {spark.table(f'{FQ}.{t}').count():>6} rows")
print("Files in volume:", [f.name for f in dbutils.fs.ls(VOLUME_PATH)])

demo_ids = ["ORD-30001", "ORD-30002", "ORD-30003", "ORD-30004", "ORD-30005"]
checks = None
for oid in demo_ids:
    df = spark.sql(f"SELECT order_id, product_name, status, days_since_delivery, eligible, refund_method, "
                   f"return_shipping_fee, estimated_refund FROM {FQ}.check_return_eligibility('{oid}')")
    checks = df if checks is None else checks.unionByName(df)
display(checks)
display(spark.sql(f"SELECT * FROM {FQ}.customer_summary('C1042')"))
display(spark.sql(f"SELECT section, content FROM {FQ}.get_policy('warranty')"))

## You are ready

If the three tables above look right (5 demo orders, Maya Chen with the **Gold** tier, 4 warranty sections), your workspace is set up.

**What you just created** in the `agents_workshop` schema:
- **Data:** `customers`, `products`, `orders`, `support_tickets`
- **Documents:** the `policy_docs` volume (return, warranty, shipping and rewards policies), plus the `policy_documents` table with the same text
- **Tools:** `check_return_eligibility(order_id)`, `customer_summary(customer_id)` and `get_policy(topic)`

**Next:** go back to the workshop guide and finish **Lab 0** (explore the catalog and create the Customer Care agent), then start **Lab 1: build the Sales Analyst agent**.

*Running this notebook again with "Start from scratch = yes" deletes and recreates the schema. Your Genie agents keep working, but the documents will be indexed again.*